# 05 — Fire weather and fuel condition
## What set the stage for the August 2024 Galičica wildfire?

In the previous practicals we mapped **where fire occurred**, **how strongly the surface changed**, and **what spectral recovery followed**.

Now we move backward in the fire lifecycle:

> **Were the weather and vegetation conditions unusually fire-conducive before and during the August 2024 event?**

We will combine two kinds of evidence:

- **ERA5-Land weather:** temperature, humidity, wind, precipitation and shallow soil moisture;
- **Sentinel-2 + land cover:** live-vegetation moisture and broad vegetation/fuel-type proxies.

The practical deliberately keeps several concepts separate:

**fire weather ≠ fuel condition ≠ fuel quantity/structure ≠ ignition probability ≠ susceptibility ≠ societal risk**

By the end you will have:

- a 2024 event-weather time series;
- a comparison against the **1991–2020 climatological baseline**;
- a simple, transparent **compound weather-stress flag count**;
- a pre-fire Sentinel-2 **NDMI** map;
- a land-cover-stratified vegetation-condition table;
- a 2026 current-season extension.

### How to work with this notebook

- Run the **core** cells from top to bottom.
- Pause at the interpretation questions before moving on.
- Most code is provided; the task is to understand the workflow, change selected parameters, and defend the interpretation.
- If a live service fails, tell a trainer rather than spending the practical debugging infrastructure.
- Stretch tasks are optional and are intended for participants who finish the core workflow early.

## 1. Imports

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

print("Python/xarray environment ready.")

## 2. Load the course AOI, EFFIS reference fire and ERA5-Land datasets

The weather inputs are prepared course subsets:

- `data/weather/era5_land_galicica_hourly_2024.nc`  
  Hourly, April–October 2024. Used for detailed event evolution.

- `data/weather/era5_land_galicica_fireseason_1991_latest.nc`  
  One standardized record per day at **12 UTC**, April–October, from 1991 through the latest available 2026 date. It also contains the preceding 24-hour precipitation.

The long record uses **1991–2020** as the climatological baseline. Years after 2020 are kept for recent comparison, not folded into the baseline.

In [ ]:
repo_root = Path.home() / "mystorage" / "fire-school"

def find_course_file(relative_path):
    candidates = [
        repo_root / relative_path,
        Path.cwd() / relative_path,
        Path.cwd().parent / relative_path,
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError(
            f"{relative_path} was not found. Run git pull and make sure the course data are present."
        )
    return path

AOI_PATH = find_course_file(Path("data/aoi/galicica_aoi.geojson"))
EFFIS_PATH = find_course_file(Path("data/effis/Galicica.gpkg"))
HOURLY_PATH = find_course_file(Path("data/weather/era5_land_galicica_hourly_2024.nc"))
FIRESEASON_PATH = find_course_file(
    Path("data/weather/era5_land_galicica_fireseason_1991_latest.nc")
)

aoi_gdf = gpd.read_file(AOI_PATH).to_crs("EPSG:4326")
aoi_geom = aoi_gdf.geometry.iloc[0]

effis = gpd.read_file(EFFIS_PATH).to_crs("EPSG:4326").copy()
effis["FIREDATE"] = pd.to_datetime(effis["FIREDATE"], errors="coerce")
if "FINALDATE" in effis.columns:
    effis["FINALDATE"] = pd.to_datetime(effis["FINALDATE"], errors="coerce")

target = effis[effis["id"].astype(str) == "240575"].copy()
if target.empty:
    raise RuntimeError("EFFIS reference polygon 240575 was not found.")

FIRE_START = pd.Timestamp(target.iloc[0]["FIREDATE"]).normalize()
if "FINALDATE" in target.columns and pd.notna(target.iloc[0]["FINALDATE"]):
    FIRE_END = pd.Timestamp(target.iloc[0]["FINALDATE"]).normalize()
else:
    FIRE_END = FIRE_START + pd.Timedelta(days=14)

hourly = xr.open_dataset(HOURLY_PATH, engine="netcdf4")
fireseason = xr.open_dataset(FIRESEASON_PATH, engine="netcdf4")

print("EFFIS fire window:", FIRE_START.date(), "→", FIRE_END.date())
print("Hourly coverage:", str(hourly.time.min().values), "→", str(hourly.time.max().values))
print("Fire-season coverage:", str(fireseason.time.min().values), "→", str(fireseason.time.max().values))
print("Fire-season variables:", ", ".join(fireseason.data_vars))

## 3. Restrict ERA5-Land to grid-cell centres inside the course AOI

ERA5-Land has a grid spacing of about 0.1°. Our downloaded rectangle is deliberately a little larger than Galičica.

For the time-series analysis we:

1. test which ERA5-Land grid-cell centres fall inside the canonical AOI;
2. mask cells outside it;
3. calculate a latitude-weighted spatial mean.

At this scale the weighting makes only a small difference, but it is a good habit for latitude/longitude grids.

In [ ]:
def coordinate_names(ds):
    lat = "latitude" if "latitude" in ds.coords else "lat"
    lon = "longitude" if "longitude" in ds.coords else "lon"
    return lat, lon

def make_aoi_mask(ds, geometry):
    lat_name, lon_name = coordinate_names(ds)
    lats = ds[lat_name].values
    lons = ds[lon_name].values

    mask = np.array([
        [geometry.covers(Point(float(lon), float(lat))) for lon in lons]
        for lat in lats
    ])

    return xr.DataArray(
        mask,
        coords={lat_name: lats, lon_name: lons},
        dims=(lat_name, lon_name),
        name="inside_aoi",
    )

def spatial_mean(ds, geometry=aoi_geom):
    lat_name, lon_name = coordinate_names(ds)
    mask = make_aoi_mask(ds, geometry)

    # Approximate latitude weighting for a regular lon/lat grid.
    weights = np.cos(np.deg2rad(ds[lat_name]))

    return (
        ds.where(mask)
        .weighted(weights)
        .mean((lat_name, lon_name))
    )

mask = make_aoi_mask(fireseason, aoi_geom)
print("ERA5-Land grid cells inside AOI:", int(mask.sum().item()))

hourly_mean = spatial_mean(hourly)
fireseason_mean = spatial_mean(fireseason)

### Resolution reminder

ERA5-Land describes the **meteorological setting**, not slope-scale wind or microclimate.

Galičica has strong elevation and terrain contrasts that a ~9 km reanalysis grid cannot fully resolve. We therefore use ERA5-Land for **regional weather context**, not for claims about exact conditions at an individual ignition point.

## 4. Detailed 2024 event weather from the hourly dataset

First we summarize the hourly series into daily diagnostics.

For each day we calculate:

- maximum AOI-mean temperature;
- minimum AOI-mean relative humidity;
- maximum AOI-mean wind speed;
- total precipitation;
- mean shallow soil moisture.

These are **descriptive diagnostics**. They are not FWI components and should not be presented as an operational danger rating.

In [ ]:
required_hourly = [
    "t2m_c",
    "rh_pct",
    "wind_speed_ms",
    "precip_hourly_mm",
    "swvl1",
]

missing = [v for v in required_hourly if v not in hourly_mean]
if missing:
    raise RuntimeError(
        f"Hourly course file is missing expected derived variables: {missing}"
    )

daily_2024 = xr.Dataset({
    "tmax_c": hourly_mean["t2m_c"].resample(time="1D").max(),
    "rhmin_pct": hourly_mean["rh_pct"].resample(time="1D").min(),
    "windmax_ms": hourly_mean["wind_speed_ms"].resample(time="1D").max(),
    "precip_mm": hourly_mean["precip_hourly_mm"].resample(time="1D").sum(min_count=1),
    "swvl1_mean": hourly_mean["swvl1"].resample(time="1D").mean(),
}).to_dataframe().reset_index()

daily_2024.head()

## 5. Inspect the weeks around the fire

The EFFIS course layer records the target event from the first week of August 2024. We display a wider window so the **antecedent conditions** are visible as well.

In [ ]:
EVENT_VIEW_START = FIRE_START - pd.Timedelta(days=35)
EVENT_VIEW_END = FIRE_END + pd.Timedelta(days=20)

event_daily = daily_2024[
    daily_2024["time"].between(EVENT_VIEW_START, EVENT_VIEW_END)
].copy()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(event_daily["time"], event_daily["tmax_c"], label="Daily max temperature")
ax.axvspan(FIRE_START, FIRE_END, alpha=0.18, label="EFFIS fire period")
ax.set_ylabel("Temperature (°C)")
ax.set_title("AOI-mean daily maximum temperature")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(event_daily["time"], event_daily["rhmin_pct"], label="Daily min RH")
ax.axvspan(FIRE_START, FIRE_END, alpha=0.18, label="EFFIS fire period")
ax.set_ylabel("Relative humidity (%)")
ax.set_title("AOI-mean daily minimum relative humidity")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(event_daily["time"], event_daily["windmax_ms"], label="Daily max wind speed")
ax.axvspan(FIRE_START, FIRE_END, alpha=0.18, label="EFFIS fire period")
ax.set_ylabel("Wind speed (m/s)")
ax.set_title("AOI-mean daily maximum 10 m wind speed")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(event_daily["time"], event_daily["precip_mm"], width=0.9, label="Daily precipitation")
ax.axvspan(FIRE_START, FIRE_END, alpha=0.18, label="EFFIS fire period")
ax.set_ylabel("Precipitation (mm/day)")
ax.set_title("Daily precipitation")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.show()

### Interpretation pause

Before looking at the climatology, discuss:

1. Was there a sustained dry period before the fire?
2. Do temperature, humidity and wind peak on exactly the same days?
3. Why might a windy day after rainfall mean something different from a windy day after several dry weeks?
4. Why is one day's weather insufficient to describe fuel condition?

## 6. Compare 2024 with the 1991–2020 climatology

The compact fire-season file uses a consistent **12 UTC snapshot** each day. This is useful for comparing years without mixing different hours.

For each calendar day we calculate the 1991–2020:

- mean;
- 10th percentile;
- 90th percentile.

We then place 2024 on top of that climatological envelope.

In [ ]:
fire_df = fireseason_mean.to_dataframe().reset_index()
fire_df["time"] = pd.to_datetime(fire_df["time"])
fire_df["year"] = fire_df["time"].dt.year
fire_df["month"] = fire_df["time"].dt.month
fire_df["month_day"] = fire_df["time"].dt.strftime("%m-%d")

baseline = fire_df[
    fire_df["year"].between(1991, 2020)
].copy()

season_2024 = fire_df[
    fire_df["year"] == 2024
].copy()

def calendar_day_climatology(df, variable):
    g = df.groupby("month_day")[variable]
    return pd.DataFrame({
        f"{variable}_mean": g.mean(),
        f"{variable}_p10": g.quantile(0.10),
        f"{variable}_p90": g.quantile(0.90),
    }).reset_index()

clim_t = calendar_day_climatology(baseline, "t2m_c")
clim_rh = calendar_day_climatology(baseline, "rh_pct")
clim_sm = calendar_day_climatology(baseline, "swvl1")

season_2024 = (
    season_2024
    .merge(clim_t, on="month_day", how="left")
    .merge(clim_rh, on="month_day", how="left")
    .merge(clim_sm, on="month_day", how="left")
)

season_2024.head()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(
    season_2024["time"],
    season_2024["t2m_c_p10"],
    season_2024["t2m_c_p90"],
    alpha=0.20,
    label="1991–2020 p10–p90",
)
ax.plot(
    season_2024["time"],
    season_2024["t2m_c_mean"],
    linestyle="--",
    label="1991–2020 mean",
)
ax.plot(
    season_2024["time"],
    season_2024["t2m_c"],
    label="2024 at 12 UTC",
)
ax.axvspan(FIRE_START, FIRE_END, alpha=0.15, label="EFFIS fire period")
ax.set_ylabel("Temperature (°C)")
ax.set_title("2024 noon temperature relative to 1991–2020")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(
    season_2024["time"],
    season_2024["rh_pct_p10"],
    season_2024["rh_pct_p90"],
    alpha=0.20,
    label="1991–2020 p10–p90",
)
ax.plot(
    season_2024["time"],
    season_2024["rh_pct_mean"],
    linestyle="--",
    label="1991–2020 mean",
)
ax.plot(
    season_2024["time"],
    season_2024["rh_pct"],
    label="2024 at 12 UTC",
)
ax.axvspan(FIRE_START, FIRE_END, alpha=0.15, label="EFFIS fire period")
ax.set_ylabel("Relative humidity (%)")
ax.set_title("2024 noon relative humidity relative to 1991–2020")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 7. Antecedent drying: 30-day precipitation

Fire-conducive conditions depend partly on what happened **before** the ignition date.

We therefore calculate a rolling 30-day precipitation total separately within each fire season. The first weeks of April are left as missing because the course file begins in April and does not contain March precipitation.

In [ ]:
fire_df = fire_df.sort_values("time").copy()

fire_df["precip_30d_mm"] = (
    fire_df
    .groupby("year")["precip_24h_mm"]
    .transform(lambda s: s.rolling(30, min_periods=25).sum())
)

baseline_aug = fire_df[
    fire_df["year"].between(1991, 2020)
    & (fire_df["month"] == 8)
].copy()

obs_2024 = fire_df[
    fire_df["year"] == 2024
].copy()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(obs_2024["time"], obs_2024["precip_30d_mm"], label="2024 rolling 30-day precipitation")
ax.axvspan(FIRE_START, FIRE_END, alpha=0.18, label="EFFIS fire period")
ax.set_ylabel("Precipitation (mm / 30 days)")
ax.set_title("Antecedent precipitation during the 2024 fire season")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 8. Put the fire-start weather in percentile context

For a compact event summary, compare the 2024 fire-start date with the distribution of **all August days from 1991–2020**.

Percentiles are intentionally shown rather than converted into a categorical danger class:

- high temperature percentile → unusually hot;
- low RH percentile → unusually dry air;
- high wind percentile → unusually windy;
- low soil-moisture percentile → unusually dry shallow soil;
- low 30-day precipitation percentile → unusually dry antecedent period.

In [ ]:
def empirical_percentile(value, reference):
    reference = pd.Series(reference).dropna()
    if reference.empty or pd.isna(value):
        return np.nan
    return 100.0 * (reference <= value).mean()

fire_start_row = obs_2024.loc[
    obs_2024["time"] == FIRE_START
]

if fire_start_row.empty:
    # The EFFIS date should be available, but nearest keeps the notebook robust.
    idx = (obs_2024["time"] - FIRE_START).abs().idxmin()
    fire_start_row = obs_2024.loc[[idx]]

r = fire_start_row.iloc[0]

metrics = [
    ("12 UTC temperature", "t2m_c", "°C", "high = more conducive"),
    ("12 UTC relative humidity", "rh_pct", "%", "low = more conducive"),
    ("12 UTC wind speed", "wind_speed_ms", "m/s", "high = more conducive"),
    ("Shallow soil moisture", "swvl1", "m³/m³", "low = drier surface"),
    ("Previous 30-day precipitation", "precip_30d_mm", "mm", "low = drier antecedent period"),
]

rows = []
for label, var, unit, interpretation in metrics:
    value = r[var]
    pct = empirical_percentile(value, baseline_aug[var])
    rows.append({
        "metric": label,
        "value": value,
        "unit": unit,
        "August_1991_2020_percentile": pct,
        "interpretation": interpretation,
    })

event_summary = pd.DataFrame(rows)
event_summary["value"] = event_summary["value"].round(2)
event_summary["August_1991_2020_percentile"] = (
    event_summary["August_1991_2020_percentile"].round(1)
)

event_summary

## 9. A transparent compound weather-stress diagnostic

Operational fire-danger systems combine weather through physically and empirically calibrated models. We will **not** reinvent those systems here.

Instead, for teaching purposes, we count five transparent conditions relative to the 1991–2020 distribution **for the same calendar month**:

- temperature ≥ monthly 90th percentile;
- RH ≤ monthly 10th percentile;
- wind ≥ monthly 90th percentile;
- shallow soil moisture ≤ monthly 10th percentile;
- rolling 30-day precipitation ≤ monthly 10th percentile.

The resulting value from **0 to 5** only answers:

> How many unusually fire-conducive weather/drying signals occurred together?

It is **not a probability of fire and not an FWI substitute**.

In [ ]:
thresholds = (
    baseline
    .assign(
        precip_30d_mm=fire_df.loc[baseline.index, "precip_30d_mm"]
    )
    .groupby("month")
    .agg(
        t90=("t2m_c", lambda s: s.quantile(0.90)),
        rh10=("rh_pct", lambda s: s.quantile(0.10)),
        wind90=("wind_speed_ms", lambda s: s.quantile(0.90)),
        sm10=("swvl1", lambda s: s.quantile(0.10)),
        p30_10=("precip_30d_mm", lambda s: s.quantile(0.10)),
    )
    .reset_index()
)

def add_compound_flags(df):
    x = df.merge(thresholds, on="month", how="left").copy()
    x["flag_hot"] = x["t2m_c"] >= x["t90"]
    x["flag_dry_air"] = x["rh_pct"] <= x["rh10"]
    x["flag_windy"] = x["wind_speed_ms"] >= x["wind90"]
    x["flag_dry_soil"] = x["swvl1"] <= x["sm10"]
    x["flag_dry_30d"] = x["precip_30d_mm"] <= x["p30_10"]

    flag_cols = [
        "flag_hot",
        "flag_dry_air",
        "flag_windy",
        "flag_dry_soil",
        "flag_dry_30d",
    ]
    x["compound_flags"] = x[flag_cols].sum(axis=1)
    return x

flags_2024 = add_compound_flags(obs_2024)

fig, ax = plt.subplots(figsize=(11, 4))
ax.step(flags_2024["time"], flags_2024["compound_flags"], where="mid")
ax.axvspan(FIRE_START, FIRE_END, alpha=0.18, label="EFFIS fire period")
ax.set_ylim(-0.2, 5.2)
ax.set_yticks(range(6))
ax.set_ylabel("Number of unusual conditions")
ax.set_title("Transparent compound weather-stress flags — 2024")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

flags_2024.sort_values(
    ["compound_flags", "time"],
    ascending=[False, True],
)[
    [
        "time",
        "t2m_c",
        "rh_pct",
        "wind_speed_ms",
        "swvl1",
        "precip_30d_mm",
        "compound_flags",
    ]
].head(12)

### Interpretation pause

Discuss:

1. Does the fire begin on the single highest-scoring day?
2. If not, why is that unsurprising?
3. Which variables reflect **short-term weather** and which partly reflect **antecedent drying**?
4. What crucial information is still missing before we can talk about ignition probability?
5. What is missing before we can talk about societal risk?

## 10. Vegetation and fuel-condition proxies from Sentinel-2

Weather is only part of the pre-fire story.

Remote sensing can help characterize vegetation that may act as fuel, but terminology matters:

- **land cover** can describe broad vegetation/fuel type and continuity;
- **NDMI** is sensitive to vegetation/canopy water content and is useful as a **live-vegetation moisture proxy**;
- **NDVI** helps describe green vegetation amount/activity.

None of these directly measures dead-fuel moisture, fuel-bed depth, fine-fuel mass or total fuel load.

We use a pre-fire window of **1 July–4 August 2024** so the composite ends before the EFFIS fire start.

In [ ]:
import ee
import folium

GEE_PROJECT_ID = os.environ.get("GEE_PROJECT_ID", "").strip()

def initialize_earth_engine():
    """Initialize Earth Engine using existing credentials or the normal auth flow."""
    try:
        if GEE_PROJECT_ID:
            ee.Initialize(project=GEE_PROJECT_ID)
        else:
            ee.Initialize()
    except Exception:
        print("Earth Engine authentication is required.")
        ee.Authenticate()
        try:
            if GEE_PROJECT_ID:
                ee.Initialize(project=GEE_PROJECT_ID)
            else:
                ee.Initialize()
        except Exception as exc:
            raise RuntimeError(
                "Earth Engine could not initialize. If your account requires a "
                "registered Google Cloud project, set the GEE_PROJECT_ID "
                "environment variable, restart the kernel, and run this cell again."
            ) from exc

initialize_earth_engine()

AOI = ee.Geometry(aoi_geom.__geo_interface__)
TARGET = ee.Geometry(target.geometry.iloc[0].__geo_interface__)

PREFIRE_START = "2024-07-01"
PREFIRE_END = "2024-08-05"
MAX_CLOUD = 60

def mask_s2_scl(img):
    scl = img.select("SCL")
    bad = (
        scl.eq(3)
        .Or(scl.eq(8))
        .Or(scl.eq(9))
        .Or(scl.eq(10))
        .Or(scl.eq(11))
    )
    return (
        img.updateMask(bad.Not())
        .select(["B4", "B8", "B11"])
    )

prefire_col = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(AOI)
    .filterDate(PREFIRE_START, PREFIRE_END)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD))
    .map(mask_s2_scl)
)

prefire = prefire_col.median().clip(AOI)
ndvi = prefire.normalizedDifference(["B8", "B4"]).rename("NDVI")
ndmi = prefire.normalizedDifference(["B8", "B11"]).rename("NDMI")

worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
)

print("Sentinel-2 scenes:", prefire_col.size().getInfo())

## 11. Map pre-fire NDMI

In [ ]:
def add_ee_layer(m, ee_image, vis_params, name):
    map_id = ee_image.getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id["tile_fetcher"].url_format,
        attr="Google Earth Engine",
        name=name,
        overlay=True,
        control=True,
    ).add_to(m)

centroid = aoi_geom.centroid
CENTER = [centroid.y, centroid.x]

m = folium.Map(location=CENTER, zoom_start=10)

folium.GeoJson(
    aoi_gdf,
    name="Course AOI",
    style_function=lambda _: {
        "color": "black",
        "weight": 2,
        "fillOpacity": 0.0,
    },
).add_to(m)

target_map = target.copy()
target_map["FIREDATE"] = target_map["FIREDATE"].dt.strftime("%Y-%m-%d")
if "FINALDATE" in target_map.columns:
    target_map["FINALDATE"] = target_map["FINALDATE"].dt.strftime("%Y-%m-%d")

folium.GeoJson(
    target_map,
    name="EFFIS 240575",
    style_function=lambda _: {
        "color": "#333333",
        "weight": 2,
        "fillOpacity": 0.0,
    },
).add_to(m)

add_ee_layer(
    m,
    ndmi,
    {
        "min": -0.2,
        "max": 0.7,
        "palette": ["8c510a", "dfc27d", "f6e8c3", "80cdc1", "01665e"],
    },
    "Pre-fire NDMI",
)

folium.LayerControl().add_to(m)
m

## 12. Compare vegetation-condition proxies by broad land-cover class

We summarize the pre-fire Sentinel-2 indices inside four broad WorldCover vegetation classes:

- tree cover;
- shrubland;
- grassland;
- cropland.

This is useful for discussing **different fuel environments**, but the classes should not be interpreted as a calibrated fuel model.

In [ ]:
LAND_COVER = {
    10: "Tree cover",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
}

pixel_area = ee.Image.pixelArea()

rows = []
for code_value, label in LAND_COVER.items():
    lc_mask = worldcover.eq(code_value)

    # Calculate area and index means separately for a clear result.
    area_m2 = (
        pixel_area
        .updateMask(lc_mask)
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=AOI,
            scale=20,
            maxPixels=1e9,
            bestEffort=True,
            tileScale=2,
        )
        .get("area")
        .getInfo()
    )

    index_stats = (
        ee.Image.cat([ndvi.updateMask(lc_mask), ndmi.updateMask(lc_mask)])
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=AOI,
            scale=20,
            maxPixels=1e9,
            bestEffort=True,
            tileScale=2,
        )
        .getInfo()
    )

    rows.append({
        "land_cover": label,
        "area_km2": area_m2 / 1e6 if area_m2 is not None else np.nan,
        "mean_NDVI": index_stats.get("NDVI"),
        "mean_NDMI": index_stats.get("NDMI"),
    })

fuel_condition = pd.DataFrame(rows)
fuel_condition[["area_km2", "mean_NDVI", "mean_NDMI"]] = (
    fuel_condition[["area_km2", "mean_NDVI", "mean_NDMI"]].round(3)
)

fuel_condition

### What can we say?

A defensible interpretation might combine:

- whether regional weather was unusually hot, dry or windy;
- whether antecedent precipitation and shallow soil moisture were low;
- which broad vegetation types were present;
- whether pre-fire live vegetation appeared relatively moist or dry spatially.

A **non-defensible** conclusion would be:

> “NDMI proves that fuel load was high and therefore caused the fire.”

NDMI does not provide that information.

## 13. Current-season extension — 2026

The long ERA5-Land file continues into 2026 so we can reuse exactly the same logic for a recent-season exercise.

We do **not** infer that a fire occurred on the days below. Instead, we identify days with multiple unusual weather/drying signals and then ask whether independent active-fire or burned-area observations show anything interesting.

That second step can be linked to the near-real-time workflow.

In [ ]:
latest_year = int(fire_df["year"].max())
recent = fire_df[fire_df["year"] == latest_year].copy()
recent_flags = add_compound_flags(recent)

print(
    f"Latest fire-season date in course file: "
    f"{recent_flags['time'].max().date()}"
)

recent_flags.sort_values(
    ["compound_flags", "time"],
    ascending=[False, False],
)[
    [
        "time",
        "t2m_c",
        "rh_pct",
        "wind_speed_ms",
        "swvl1",
        "precip_30d_mm",
        "compound_flags",
    ]
].head(15)

### 2026 mini-exercise

Choose one high-compound-signal day from the table and investigate:

1. Was there a VIIRS/MODIS active-fire detection in or near the AOI?
2. Was an EFFIS event later mapped?
3. What did Sentinel-2 look like before and after?
4. If there was **no** observed fire, what does that teach us about the difference between fire-conducive weather and actual fire occurrence?

Do not treat absence from one fire product as proof that no fire occurred.

## 14. Optional stretch tasks

Choose one:

### A — Change the climatological reference
Compare 1991–2020 with 2001–2020. Do the percentile rankings change?

### B — Antecedent drought window
Compare 7-, 14- and 30-day precipitation totals. Which one best separates the 2024 fire period from surrounding weeks?

### C — Elevation
Use a DEM to compare the ERA5-Land grid with Galičica's elevation range. Where might coarse-grid weather be least representative?

### D — Historical vegetation moisture
Calculate the same July–early-August NDMI composite for 2019–2023 and ask whether 2024 was spectrally unusual.

### E — Operational FWI
Implement or use a validated Canadian FWI calculation with the correct meteorological time conventions. Compare it with the transparent flag-count exercise above.

### F — Current season
Repeat the 2026 exercise for a recent fire elsewhere in the Western Balkans.

## 15. Output for the Galičica capstone

Keep:

- one **2024 weather-vs-climatology figure**;
- the fire-start percentile table;
- the compound weather-stress figure or table;
- the pre-fire NDMI map;
- the land-cover / vegetation-condition table;
- **three defensible findings**;
- **one limitation**;
- **one management-relevant interpretation**.

A strong conclusion should explicitly separate:

**weather conditions → vegetation/fuel-condition proxies → observed fire**

from the stronger claims we have **not** established:

**ignition cause, fuel load, fire probability, susceptibility or risk.**

The next practical combines several relatively static environmental factors into an interpretable **fire-susceptibility** workflow.